# 01. Data Cleaning & Multi-City Preprocessing Pipeline

This notebook demonstrates the end-to-end data ingestion and preprocessing pipeline for the **Intelligent Airbnb Search & Ranking Engine**.

### Objectives:
1. **Auto-discover** all available city folders inside `data/raw/`.
2. Ingest `listings.csv`, `reviews.csv`, `calendar.csv`, and `neighbourhoods.csv`.
3. Handle schema discrepancies gracefully (e.g. minimal vs full schema).
4. Clean numerical prices, sanitize textual attributes, and parse nested amenities.
5. Construct the rich composite `listing_text` for embedding generation.
6. Save clean, compressed parquet datasets into `data/processed/`.

In [ ]:
import sys
from pathlib import Path

# Set root path
ROOT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT_DIR))

import pandas as pd
import numpy as np
import config
from src.preprocessing import (
    load_all_listings,
    clean_listings,
    process_reviews,
    process_calendar,
    load_neighbourhoods,
    compute_and_save_centroids,
    merge_and_save,
    print_summary,
)

print(f"Configured Raw Directory: {config.RAW_DIR}")
print(f"Auto-discovered Cities: {config.discover_cities()}")

## 1. Load Raw Multi-City Listings

In [ ]:
raw_df = load_all_listings()
print(f"Total Raw Listings: {len(raw_df):,}")
print(f"City breakdown:\n{raw_df['city'].value_counts()}")
raw_df.head(3)

## 2. Clean & Preprocess Listings
- Parse string prices (`$150.00` -> `150.0`)
- Impute missing ratings per city using city-level medians
- Parse amenities list into top indicator columns
- Generate rich `listing_text`

In [ ]:
clean_df = clean_listings(raw_df)
print(f"Cleaned Listings: {len(clean_df):,} (Kept {len(clean_df)/len(raw_df):.1%})")
clean_df[['city', 'name', 'price_usd', 'room_type', 'listing_text']].head(3)

## 3. Review & Calendar Feature Aggregation

In [ ]:
reviews_agg = process_reviews()
calendar_agg = process_calendar()
neighbourhoods_df = load_neighbourhoods()

print(f"Review records aggregated: {len(reviews_agg):,}")
print(f"Calendar records aggregated: {len(calendar_agg):,}")

## 4. Merge Everything and Save Clean Parquet Dataset

In [ ]:
compute_and_save_centroids(clean_df)
final_df = merge_and_save(clean_df, reviews_agg, calendar_agg)
print_summary(final_df)
print(f"Final Processed Listings saved to: {config.LISTINGS_CLEAN_PATH}")